# 🐍 Python from Scratch — Module 9: Testing Your Code

### How do you know your function actually works?

So far, you've checked whether code works by eyeballing `print()` output. That's fine
for one function in a notebook, but it doesn't scale — the more code you have, the
harder it is to manually re-check everything after every change. Tests are a way to
automatically and repeatably check "does this still work".

## Table of Contents

1. [Why test code](#sec1)
2. [`assert` — the simplest test](#sec2)
3. [The `unittest` module — the basics](#sec3)
4. [The most useful `assert*` methods](#sec4)
5. [Testing exceptions](#sec5)
6. [Organizing tests, and `pytest`](#sec6)
7. [What to actually test — edge cases](#sec7)
8. [Fun fact: the testing pyramid](#sec8)
9. [Module summary](#sec9)
10. [Exercises](#sec10)

---

<a id="sec1"></a>
## 1. Why test code

Imagine the `calculate_bmi()` function from Module 4, used inside a bigger program.
Another developer (or you, six months later) tweaks something in that function and
accidentally breaks the formula. Without tests, you'll only notice once something
breaks in production - or maybe not at all. With tests: you run one command, and within
a second you know something's wrong, before anyone else ever sees it.

<a id="sec2"></a>
## 2. `assert` — the simplest test

The `assert expression` statement does nothing if `expression` is true (`True`), and
raises an `AssertionError` if it's false. This is the foundation that more elaborate
testing tools are built on.

In [ ]:
def add(a, b):
    return a + b

assert add(2, 3) == 5          # passes silently
assert add(-1, 1) == 0         # also passes

# This raises an AssertionError, because the function is (deliberately) buggy:
def buggy_add(a, b):
    return a + b + 1   # bug: an extra +1

try:
    assert buggy_add(2, 3) == 5
except AssertionError:
    print("Test failed - the function returned the wrong result!")

<a id="sec3"></a>
## 3. The `unittest` module — the basics

`unittest` is a built-in module for writing tests in a more organized way. Tests are
grouped inside a class that inherits from `unittest.TestCase`, and each method starting
with `test_` is a separate, independent test.

In [ ]:
%%writefile calculator.py
def add(a, b):
    return a + b

def subtract(a, b):
    return a - b

In [ ]:
%%writefile test_calculator.py
import unittest
from calculator import add, subtract

class TestCalculator(unittest.TestCase):
    def test_addition(self):
        self.assertEqual(add(2, 3), 5)

    def test_addition_with_negatives(self):
        self.assertEqual(add(-1, -1), -2)

    def test_subtraction(self):
        self.assertEqual(subtract(5, 3), 2)

if __name__ == "__main__":
    unittest.main()

In [ ]:
# Running all the tests in the file (looks the same in a real terminal):
!python -m unittest test_calculator.py -v

> 💡 **Where the `test_...` names come from**
>
> `unittest` (and `pytest`) automatically recognize methods and files starting with `test_` as tests to run - that's a convention, not a coincidence. It lets you keep production code and tests in separate files, and the tool finds what to run on its own.

<a id="sec4"></a>
## 4. The most useful `assert*` methods

`unittest.TestCase` has a whole family of `assert*` methods, more readable than a bare
`assert` - partly because on failure, they show exactly what was expected versus what
actually happened.

| Method | Checks |
|---|---|
| `assertEqual(a, b)` | that `a == b` |
| `assertNotEqual(a, b)` | that `a != b` |
| `assertTrue(x)` | that `x` is truthy |
| `assertFalse(x)` | that `x` is falsy |
| `assertIsNone(x)` | that `x` is `None` |
| `assertIn(a, b)` | that `a` is in `b` (a list, a string...) |
| `assertRaises(Error, ...)` | that a call raises the given exception |

<a id="sec5"></a>
## 5. Testing exceptions

In Module 5, you learned to raise errors with `raise`. It's also worth testing that a
function **actually** raises an exception when it should - `assertRaises` checks
exactly that, most often used as a context manager (`with`).

In [ ]:
%%writefile account.py
class BankAccount:
    def __init__(self):
        self._balance = 0

    def deposit(self, amount):
        self._balance += amount

    def withdraw(self, amount):
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount

In [ ]:
%%writefile test_account.py
import unittest
from account import BankAccount

class TestBankAccount(unittest.TestCase):
    def test_deposit_increases_balance(self):
        account = BankAccount()
        account.deposit(100)
        self.assertEqual(account._balance, 100)

    def test_withdraw_over_balance_raises_error(self):
        account = BankAccount()
        account.deposit(50)
        with self.assertRaises(ValueError):
            account.withdraw(100)

if __name__ == "__main__":
    unittest.main()

In [ ]:
!python -m unittest test_account.py -v

<a id="sec6"></a>
## 6. Organizing tests, and `pytest`

In real projects, tests usually live in a separate `tests/` folder, with one
`test_*.py` file per production module. `pytest` is a popular third-party package
(`pip install pytest` - remember Module 7?) that does the same job as `unittest`, but
with simpler syntax - plain `assert`, no classes or inheritance needed.

```python
# Same idea as the section 3 example, but pytest-style (file test_calculator_pytest.py):
from calculator import add, subtract

def test_addition():
    assert add(2, 3) == 5

def test_subtraction():
    assert subtract(5, 3) == 2
```

You run this with the `pytest` command in a terminal (after `pip install pytest`) - the
tool automatically finds every `test_*.py` file and every `test_*` function in the
current folder.

<a id="sec7"></a>
## 7. What to actually test — edge cases

A good test doesn't just check the "typical" case. It's worth thinking about **edge
cases**: an empty list, the number 0 or a negative number, a very large value, an empty
string or one made of nothing but spaces. That's usually where bugs hide.

In [ ]:
def average(numbers):
    return sum(numbers) / len(numbers)

print(average([1, 2, 3]))   # the typical case - works fine

# What about an empty list?
try:
    print(average([]))
except ZeroDivisionError:
    print("An empty list causes division by zero - worth testing and handling!")

> ⚠️ **A test that can never fail is worthless**
>
> Writing `assertEqual(2 + 2, 4)` inside a test for your function checks nothing - a test needs to actually call your code and check its result, ideally with several different, sensibly chosen inputs.

<a id="sec8"></a>
## 8. Fun fact: the testing pyramid

In larger projects, tests are usually split into layers.

> 💡 **Fun fact**
>
> The «testing pyramid» is a popular model: at the base, lots of fast **unit tests** (like the ones in this module - a single function, in isolation), above that fewer **integration tests** (checking that several pieces work correctly together), and at the top a small number of slower **end-to-end tests** (checking the whole program from a user's point of view). The lower down the pyramid, the faster and cheaper to maintain a test should be.

<a id="sec9"></a>
## 9. Module summary

By now it should be clear:

- how `assert` works, and how it differs from `unittest`'s `assertEqual`,
- how to organize tests inside a `unittest.TestCase` class,
- the most useful `assert*` methods (`assertEqual`, `assertTrue`, `assertRaises`...),
- how to test that a function correctly raises an exception,
- the difference between `unittest` and `pytest`,
- why it's worth testing edge cases, not just the typical scenario.

The final module in this series: **working with APIs and an introduction to pandas** —
fetching data from the internet and analyzing it, tying together everything you've
learned so far.

<a id="sec10"></a>
## 10. Exercises

A few tasks revisit functions from earlier modules - this time to write tests for them,
rather than just calling them.

> 📝 **Exercise 1: Tests for `is_even`**
>
> Create a file `parity.py` with the `is_even(n)` function from Module 4. Write a `unittest.TestCase` for it with at least two tests: an even number and an odd number. Run the tests with `!python -m unittest`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile parity.py
# def is_even(n):
#     return n % 2 == 0

# %%writefile test_parity.py
# import unittest
# from parity import is_even
#
# class TestParity(unittest.TestCase):
#     def test_even_number(self):
#         self.assertTrue(is_even(4))
#
#     def test_odd_number(self):
#         self.assertFalse(is_even(7))
#
# if __name__ == "__main__":
#     unittest.main()

# !python -m unittest test_parity.py -v
```
</details>

> 📝 **Exercise 2: Tests for the BMI calculator**
>
> Create a file `bmi.py` with the `calculate_bmi(weight, height)` function from Module 4. Write a test checking that for `weight=70, height=1.75` the result is approximately `22.86` - use `assertAlmostEqual(result, 22.86, places=2)`, since comparing floating-point numbers with `==` can be unreliable.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile bmi.py
# def calculate_bmi(weight, height):
#     return weight / height ** 2

# %%writefile test_bmi.py
# import unittest
# from bmi import calculate_bmi
#
# class TestBmi(unittest.TestCase):
#     def test_typical_case(self):
#         result = calculate_bmi(70, 1.75)
#         self.assertAlmostEqual(result, 22.86, places=2)
#
# if __name__ == "__main__":
#     unittest.main()

# !python -m unittest test_bmi.py -v
```

Hint: `assertAlmostEqual` compares numbers with a tolerance, which matters for floating-point numbers - `0.1 + 0.2 == 0.3` evaluates to `False` in Python because of how fractional numbers are stored in memory.
</details>

> 📝 **Exercise 3: Testing an edge case — an empty list**
>
> Write a function `safe_average(numbers)` that returns `None` for an empty list instead of crashing with `ZeroDivisionError` (hint: `if not numbers: return None`). Write TWO tests: one for a normal list, and one checking the behavior for an empty list.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile average.py
# def safe_average(numbers):
#     if not numbers:
#         return None
#     return sum(numbers) / len(numbers)

# %%writefile test_average.py
# import unittest
# from average import safe_average
#
# class TestAverage(unittest.TestCase):
#     def test_typical_list(self):
#         self.assertEqual(safe_average([1, 2, 3]), 2)
#
#     def test_empty_list(self):
#         self.assertIsNone(safe_average([]))
#
# if __name__ == "__main__":
#     unittest.main()

# !python -m unittest test_average.py -v
```
</details>

> 📝 **Exercise 4: Testing a raised exception**
>
> Given the `BankAccount` class from section 5, write an additional test checking that trying to deposit a **negative** amount raises a `ValueError` - to do this, first add that validation to the `deposit()` method, then write a test for it with `assertRaises`.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# In account.py, the deposit() method should start with:
# def deposit(self, amount):
#     if amount < 0:
#         raise ValueError("Deposit amount cannot be negative")
#     self._balance += amount

# In the test:
# def test_negative_deposit_raises_error(self):
#     account = BankAccount()
#     with self.assertRaises(ValueError):
#         account.deposit(-50)
```
</details>

> 📝 **Exercise 5: Tests for password validation**
>
> Given the `set_password(password)` function from Module 5 (raises `ValueError` if the password is shorter than 8 characters), write two tests: one for a valid password (checking the returned value), and one for a too-short password (checking that it raises a `ValueError`).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile password.py
# def set_password(password):
#     if len(password) < 8:
#         raise ValueError("Password must be at least 8 characters long")
#     return "Password set"

# %%writefile test_password.py
# import unittest
# from password import set_password
#
# class TestPassword(unittest.TestCase):
#     def test_valid_password(self):
#         self.assertEqual(set_password("secure123"), "Password set")
#
#     def test_too_short_password(self):
#         with self.assertRaises(ValueError):
#             set_password("abc")
#
# if __name__ == "__main__":
#     unittest.main()
```
</details>

> 📝 **Exercise 6: Tests for `is_palindrome`**
>
> Given the `is_palindrome(s)` function from Module 7, write at least three tests: a simple palindrome (e.g. `"kayak"`), a non-palindrome (e.g. `"python"`), and a mixed-case case (e.g. `"Kayak"` should also count as a palindrome).

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile text_util.py
# def is_palindrome(s):
#     s = s.lower()
#     return s == s[::-1]

# %%writefile test_text_util.py
# import unittest
# from text_util import is_palindrome
#
# class TestPalindrome(unittest.TestCase):
#     def test_simple_palindrome(self):
#         self.assertTrue(is_palindrome("kayak"))
#
#     def test_not_a_palindrome(self):
#         self.assertFalse(is_palindrome("python"))
#
#     def test_mixed_case(self):
#         self.assertTrue(is_palindrome("Kayak"))
#
# if __name__ == "__main__":
#     unittest.main()
```
</details>

> 📝 **Exercise 7: A manually parameterized test**
>
> Given the `is_prime(n)` function from Module 7, write ONE test that, in a loop, checks a list of known primes `[2, 3, 5, 7, 11, 13]` (all should give `True`) and a list of known composites `[4, 6, 8, 9, 10]` (all should give `False`) - without writing a separate `test_` method for each number.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# %%writefile numbers_util.py
# def is_prime(n):
#     if n < 2:
#         return False
#     for divisor in range(2, int(n ** 0.5) + 1):
#         if n % divisor == 0:
#             return False
#     return True

# %%writefile test_numbers_util.py
# import unittest
# from numbers_util import is_prime
#
# class TestPrimeNumbers(unittest.TestCase):
#     def test_prime_numbers(self):
#         for number in [2, 3, 5, 7, 11, 13]:
#             self.assertTrue(is_prime(number), f"{number} should be prime")
#
#     def test_composite_numbers(self):
#         for number in [4, 6, 8, 9, 10]:
#             self.assertFalse(is_prime(number), f"{number} should not be prime")
#
# if __name__ == "__main__":
#     unittest.main()
```

Hint: `assertTrue`/`assertFalse`'s optional second argument is a message shown when the test fails - very handy inside a loop, so you know exactly WHICH number actually failed the test.
</details>

> 🔥 **Exercise 8 (challenge): TDD - test first, code second**
>
> This time, flip the order: write the tests FIRST for a function `count_occurrences(text, word)` (should return how many times the given word appears in the text, case-insensitively), covering: a typical case, a word that doesn't appear (result 0), and an empty text. Only AFTER writing the tests, implement the function so that all of them pass.

<details>
<summary><b>👉 Click to reveal a sample solution</b></summary>

```python
# Step 1 - tests BEFORE the implementation (they will fail until the function exists):
# %%writefile test_word_count.py
# import unittest
# from word_count import count_occurrences
#
# class TestWordCount(unittest.TestCase):
#     def test_typical_case(self):
#         text = "python is great, I am learning python, I love Python"
#         self.assertEqual(count_occurrences(text, "python"), 3)
#
#     def test_word_not_present(self):
#         self.assertEqual(count_occurrences("some text", "missing"), 0)
#
#     def test_empty_text(self):
#         self.assertEqual(count_occurrences("", "anything"), 0)
#
# if __name__ == "__main__":
#     unittest.main()

# Step 2 - only now the implementation, written to make the tests pass:
# %%writefile word_count.py
# def count_occurrences(text, word):
#     words = text.lower().split()
#     return words.count(word.lower())

# !python -m unittest test_word_count.py -v
```

This is the essence of TDD (Test-Driven Development) - tests define what you expect from the code BEFORE that code exists. It forces you to think about edge cases upfront, rather than as an afterthought.
</details>

---

### What's next?

If the TDD exercise (tests before code) went reasonably smoothly, you have a solid
foundation for writing code you can trust. The tenth and final module in this series
ties everything together: working with an external API (`requests`) and an introduction
to data analysis (`pandas`).